# Step 7-1: 문장 분할 (ABSA 전처리)

**방법 C — 하이브리드 ABSA의 첫 단계**

리뷰를 문장 단위로 쪼개서, 각 문장이 다루는 측면(aspect)과 감성을 분리 추출할 수 있도록 준비한다.

## 왜 문장 분할이 필요한가

리뷰 단위 감성분석은 한 리뷰의 전체 톤만 잡기 때문에, "기모는 따뜻한데 사이즈는 작아요" 같은 mixed 리뷰에서 측면별 감성을 분리할 수 없다.
문장 단위로 쪼개면:
- "기모는 따뜻한데" → (기모, 따뜻 측면, 긍정)
- "사이즈는 작아요" → (사이즈 측면, 부정)
이렇게 측면별로 정확히 추출 가능.

## 입력 / 출력
| 파일 | 내용 |
|------|------|
| `embedding_input.parquet` | 정제된 리뷰 (step3 임베딩 입력과 동일) |
| `step5_1_cluster_labels.npy` | HDBSCAN 토픽 라벨 |
| → `step7_1_sentences.parquet` | 문장 단위로 분할된 데이터 (한 리뷰 → N행) |

## 분할 전략 (2단 분할)
1. **Kiwi `split_into_sents`** — 종결어미 기준 1차 분할 (마침표·줄바꿈·"-요/-다" 등)
2. **역접 접속사 분할** — `-는데/지만/그러나/그런데/근데/하지만/다만/반면` 직후로 2차 분할
   - mixed 리뷰의 측면 분리가 핵심
3. **2자 미만 문장 필터링**

## 결과 데이터 구조
| 컬럼 | 설명 |
|---|---|
| `리뷰번호` | 원본 리뷰 ID |
| `sent_id` | 0,1,2... (한 리뷰 내 문장 순번) |
| `sentence` | 분할된 문장 |
| `n_sents` | 그 리뷰의 총 문장 수 |
| `topic_id` | HDBSCAN 토픽 (-1 = outlier) |
| `평점`, `goodsNo`, `작성일` | 메타 정보 |

In [1]:
!pip install -q kiwipiepy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 9.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 18.4 MB/s eta 0:00:00


In [2]:
from google.colab import drive

import os
import re
import time
import numpy as np
import pandas as pd
from kiwipiepy import Kiwi
from tqdm.auto import tqdm

In [3]:
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/sparta/tp4/review_analysis/data/'
INPUT_PATH = OUTPUT_DIR + 'embedding_input.parquet'

print(f'INPUT     : {INPUT_PATH}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

Mounted at /content/drive
INPUT     : /content/drive/MyDrive/sparta/tp4/review_analysis/data/embedding_input.parquet
OUTPUT_DIR: /content/drive/MyDrive/sparta/tp4/review_analysis/data/


## 7-1-1. 데이터 로드

`embedding_input.parquet`(중복 제거된 리뷰)과 `04_cluster_labels.npy`(토픽 라벨)을 로드.

> **주의**: 이 두 파일은 길이가 정확히 같아야 함. step3 임베딩 입력 = step5 클러스터링 결과.

In [4]:
# 정제된 리뷰 로드
df = pd.read_parquet(INPUT_PATH)
df['리뷰내용_clean'] = df['리뷰내용_clean'].fillna('').astype(str)
df = df[df['리뷰내용_clean'].str.len() > 0].reset_index(drop=True)

# 클러스터 라벨 로드
cluster_labels = np.load('/content/drive/MyDrive/sparta/tp4/review_analysis/data/step5_1_cluster_labels.npy')

# 길이 검증
assert len(df) == len(cluster_labels), \
    f'길이 불일치! df={len(df)} vs cluster_labels={len(cluster_labels)}'

df['topic_id'] = cluster_labels

print(f'리뷰 수: {len(df):,}건')
print(f'토픽 수 (outlier 제외): {len(set(cluster_labels)) - 1}개')
print(f'Outlier(-1) 비율: {(cluster_labels == -1).mean()*100:.1f}%')
print(f'\n[컬럼 목록]')
print(df.columns.tolist())

리뷰 수: 598,301건
토픽 수 (outlier 제외): 68개
Outlier(-1) 비율: 31.5%

[컬럼 목록]
['리뷰번호', '리뷰내용_clean', 'goodsNo', '리뷰타입', '평점', '작성일', '사이즈', '화면대비색감', '퀄리티', '구김', '두께감', '신축성', '색감', '보온성', '퀄리티_점수', '보온성_점수', '신축성_점수', '두께감_점수', '구김_점수', '사이즈_편차절대', '화면대비색감_편차절대', '색감_편차절대', '만족도_응답여부', 'laugh_count', 'cry_count', 'exclamation_count', 'question_count', 'emoji_count', 'has_repetition', 'text_length_orig', '성별', '구매사이즈', '구매상세', '사진유무', '도움돼요', '체험단', 'topic_id']


## 7-1-2. Kiwi 토크나이저 + 보호어 등록

**중요**: step6에서 등록한 패션 도메인 보호어를 동일하게 등록해야 한다.
이렇게 해야 `오버핏`, `시보리` 같은 복합어가 문장 분할 시 분리되지 않음.

> 가능하면 step6 노트북에서 `PROTECTED_WORDS`를 별도 파일(`patterns.json`)로 저장해두고 양쪽에서 import하는 게 동기화 측면에서 안전. 일단 여기서는 직접 정의.

In [5]:
import json
with open('/content/drive/MyDrive/sparta/tp4/review_analysis/data/protected_words.json', 'r', encoding='utf-8') as f:
    PROTECTED_WORDS = json.load(f)

all_protected = sum(PROTECTED_WORDS.values(), [])
print(f'전체 보호어: {len(all_protected)}개')
print(f'카테고리: {list(PROTECTED_WORDS.keys())}')

kiwi = Kiwi()
for word in all_protected:
    kiwi.add_user_word(word, 'NNG', score=5)
print(f'Kiwi 사용자 사전 등록 완료')

전체 보호어: 104개
카테고리: ['fit', 'detail', 'fabric', 'color', 'value', 'size']
Kiwi 사용자 사전 등록 완료


## 7-1-3. 분할 함수 정의 (Kiwi 형태소 분석 기반)

### 핵심 아이디어
Kiwi는 어미 결합 시 표면형이 변형돼도 **형태소 단위로는 일관된 form을 반환**한다.

| 표면형 | Kiwi form | 태그 |
|---|---|---|
| 예쁜데 | 예쁘 + **ᆫ데** | VA + **EC** |
| 좋은데 | 좋 + **은데** | VA + **EC** |
| 두꺼운데 | 두껍 + **은데** | VA-I + **EC** |
| 먹는데 | 먹 + **는데** | VV + **EC** |

→ 정규식은 `예쁜데`, `좋은데`, `두꺼운데`를 각각 다르게 매칭해야 하지만, Kiwi는 form 3개(`ᆫ데`, `은데`, `는데`)만 잡으면 끝.

### 의존명사 '데' 구분
일부 케이스(`따뜻한데`, `비싼데` 등)는 Kiwi가 EC 대신 ETM(`ᆫ`) + NNB(`데`)로 오분석한다.
또한 진짜 의존명사 '데'(`아픈 데가 있어요`)도 같은 패턴으로 분석된다.
**휴리스틱**: NNB('데') 다음이 조사(JKS/JX/JKB)면 진짜 의존명사 → 분할 X.

In [6]:
# 대조/양보 의미의 연결어미 (Kiwi form 기준)
CONTRAST_EC = {
    'ᆫ데', '은데', '는데',          # -ㄴ데/-은데/-는데 (가장 빈번)
    '지만',                          # -지만
    '더라도', '어도', '아도', '여도', # 양보
    '으나', '나',                    # -으나/-나
}

# 접속 부사 (form 기준 — tag 무관, '반면'이 NNG든 MAJ든 잡힘)
CONTRAST_ADV = {
    '그러나', '그런데', '근데', '하지만', '다만', '반면',
    '그렇지만', '그래도', '그럼에도', '오히려', '도리어', '반대로',
}

# ETM + NNB('데') 패턴에서 연결어미로 인정할 ETM form
ETM_NDE = {'ᆫ', '은', '는'}  # 'ᆯ'(미래)은 제외

MIN_SENT_LEN = 2  # 2자 미만 문장 제거


def split_sentences(text: str) -> list:
    """Kiwi 형태소 분석 기반 정밀 문장 분할.

    분할 단계:
    1) Kiwi split_into_sents — 종결어미(EF) 기준
    2) EC 연결어미 직후 — CONTRAST_EC form 매칭
    3) ETM + NNB('데') 보완 — 단, 다음이 조사면 의존명사로 보존
    4) 접속 부사 직전 — CONTRAST_ADV form 매칭 (첫 토큰 제외)
    """
    if not text or not isinstance(text, str):
        return []

    # 1차 분할
    sents = [s.text.strip() for s in kiwi.split_into_sents(text)]

    final_sents = []
    for sent in sents:
        if not sent:
            continue

        try:
            tokens = kiwi.tokenize(sent)
        except Exception:
            final_sents.append(sent)
            continue

        split_points = []
        n = len(tokens)

        for i, tok in enumerate(tokens):
            # (1) EC 연결어미 (정상 분석)
            if tok.tag == 'EC' and tok.form in CONTRAST_EC:
                split_points.append(tok.start + tok.len)

            # (2) ETM + NNB('데') 오분석 보완
            elif (tok.tag == 'ETM' and tok.form in ETM_NDE
                  and i + 1 < n
                  and tokens[i+1].tag == 'NNB' and tokens[i+1].form == '데'):
                # 마지막 토큰이면 의존명사일 가능성 → 분할 X
                if i + 2 >= n:
                    continue
                # 다음 토큰이 조사면 진짜 의존명사 → 분할 X
                if tokens[i+2].tag.startswith('J'):
                    continue
                # 그 외 → 연결어미로 간주 → 분할
                split_points.append(tokens[i+1].start + tokens[i+1].len)

            # (3) 접속 부사 (tag 무관, form만 매칭, 첫 토큰 제외)
            elif i > 0 and tok.form in CONTRAST_ADV:
                split_points.append(tok.start)

        if not split_points:
            final_sents.append(sent)
            continue

        split_points = sorted(set(split_points))
        prev = 0
        for sp in split_points:
            chunk = sent[prev:sp].strip()
            if len(chunk) >= MIN_SENT_LEN:
                final_sents.append(chunk)
            prev = sp
        last = sent[prev:].strip()
        if len(last) >= MIN_SENT_LEN:
            final_sents.append(last)

    return final_sents


# ─────────────────────────────────────────────
# 검증 테스트
# ─────────────────────────────────────────────
test_cases = [
    # ㄴ받침 합쳐진 케이스
    ('예쁜데 비싸요', 2),
    ('큰데 헐렁해요', 2),
    ('따뜻한데 사이즈가 작아요', 2),
    ('비싼데 만족해요', 2),
    # 받침 있는 형용사
    ('재질은 좋은데 작아요', 2),
    # ㅂ불규칙
    ('두꺼운데 따뜻해요', 2),
    ('어려운데 도전할만해요', 2),
    # 동사 / 명사
    ('먹는데 맛없어요', 2),
    ('오버핏인데 핏이 좋아요', 2),
    # 다양한 어미
    ('비싸지만 만족해요', 2),
    ('작아도 괜찮아요', 2),
    # 접속 부사
    ('예뻐요. 근데 작아요', 2),
    ('예뻐요 반면에 비싸요', 2),
    ('편해요 도리어 좋아요', 2),
    # 의존명사 '데' (분할 X)
    ('아픈 데가 있어요', 1),
    ('갈 데도 없어요', 1),
    # 병렬 (분할 X)
    ('색감이 예쁘고 사이즈도 딱맞아요', 1),
    # 복잡
    ('예쁜데 비싸지만 만족해요', 3),
    ('갈 데도 없는데 어쩌지', 2),
]

print('[분할 검증]')
correct = 0
for text, expected in test_cases:
    parts = split_sentences(text)
    actual = len(parts)
    ok = '[OK]' if actual == expected else '[XX]'
    if actual == expected:
        correct += 1
    print(f'{ok} (n={actual}, 기대={expected}) {text}')
    if actual != expected:
        for p in parts:
            print(f'      → {p!r}')

print(f'\n정확도: {correct}/{len(test_cases)} ({correct/len(test_cases)*100:.0f}%)')

[분할 검증]
[OK] (n=2, 기대=2) 예쁜데 비싸요
[OK] (n=2, 기대=2) 큰데 헐렁해요
[OK] (n=2, 기대=2) 따뜻한데 사이즈가 작아요
[OK] (n=2, 기대=2) 비싼데 만족해요
[OK] (n=2, 기대=2) 재질은 좋은데 작아요
[OK] (n=2, 기대=2) 두꺼운데 따뜻해요
[OK] (n=2, 기대=2) 어려운데 도전할만해요
[OK] (n=2, 기대=2) 먹는데 맛없어요
[OK] (n=2, 기대=2) 오버핏인데 핏이 좋아요
[OK] (n=2, 기대=2) 비싸지만 만족해요
[OK] (n=2, 기대=2) 작아도 괜찮아요
[OK] (n=2, 기대=2) 예뻐요. 근데 작아요
[OK] (n=2, 기대=2) 예뻐요 반면에 비싸요
[OK] (n=2, 기대=2) 편해요 도리어 좋아요
[OK] (n=1, 기대=1) 아픈 데가 있어요
[OK] (n=1, 기대=1) 갈 데도 없어요
[OK] (n=1, 기대=1) 색감이 예쁘고 사이즈도 딱맞아요
[OK] (n=3, 기대=3) 예쁜데 비싸지만 만족해요
[OK] (n=2, 기대=2) 갈 데도 없는데 어쩌지

정확도: 19/19 (100%)


## 7-1-4. 전체 데이터에 적용

62만 건 x 평균 1.5~2문장 → 약 100만 문장 예상.
형태소 분석은 정규식보다 느려서 **15~25분** 정도 예상.

In [7]:
# 전체 적용
print('문장 분할 중... (~15-25분, Kiwi 형태소 분석)')
t0 = time.time()

tqdm.pandas()
df['sentences'] = df['리뷰내용_clean'].progress_apply(split_sentences)
df['n_sents'] = df['sentences'].apply(len)

elapsed = (time.time() - t0) / 60
print(f'\n분할 완료: {elapsed:.1f}분')

print(f'\n[문장 수 분포]')
print(df['n_sents'].describe())

total_sents = int(df['n_sents'].sum())
print(f'\n총 문장 수: {total_sents:,}')
print(f'평균 문장/리뷰: {total_sents/len(df):.2f}')
n_multi = (df['n_sents'] >= 2).sum()
print(f'2문장 이상 분할된 리뷰: {n_multi:,}건 ({n_multi/len(df)*100:.1f}%)')

문장 분할 중... (~15-25분, Kiwi 형태소 분석)


  0%|          | 0/598301 [00:00<?, ?it/s]


분할 완료: 27.9분

[문장 수 분포]
count    598301.000000
mean          2.569210
std           1.400935
min           1.000000
25%           2.000000
50%           2.000000
75%           3.000000
max          60.000000
Name: n_sents, dtype: float64

총 문장 수: 1,537,161
평균 문장/리뷰: 2.57
2문장 이상 분할된 리뷰: 507,885건 (84.9%)


## 7-1-5. 문장 단위로 explode

In [8]:
# 보존할 메타 컬럼 (실제 존재하는 것만)
CANDIDATE_META = ['리뷰번호', 'topic_id', 'n_sents',
                  '별점', '브랜드', '카테고리', '상품번호', '날짜',
                  'is_valid_for_topic']

META_COLS = [c for c in CANDIDATE_META if c in df.columns]
print(f'보존 메타 컬럼: {META_COLS}')

# explode
df_sents = (
    df[META_COLS + ['sentences']]
    .explode('sentences')
    .reset_index(drop=True)
    .rename(columns={'sentences': 'sentence'})
)

# 안전장치: 빈 문장 제거
df_sents = df_sents[df_sents['sentence'].notna()].reset_index(drop=True)
df_sents = df_sents[df_sents['sentence'].str.len() >= MIN_SENT_LEN].reset_index(drop=True)

# sent_id (한 리뷰 내 문장 순번)
df_sents['sent_id'] = df_sents.groupby('리뷰번호').cumcount()

# 컬럼 순서 정리
front_cols = [c for c in ['리뷰번호', 'sent_id', 'sentence', 'n_sents', 'topic_id'] if c in df_sents.columns]
other_cols = [c for c in df_sents.columns if c not in front_cols]
df_sents = df_sents[front_cols + other_cols]

print(f'\nexplode 결과: {len(df_sents):,}행')
print(f'\n[샘플 10건]')
df_sents.head(10)

보존 메타 컬럼: ['리뷰번호', 'topic_id', 'n_sents']

explode 결과: 1,531,645행

[샘플 10건]


,리뷰번호,sent_id,sentence,n_sents,topic_id
0,34612047,0,요즘 입기 좋은 것 같아요,2,67
1,34612047,1,무난무난하게 잘 입고있습니다,2,67
2,51564285,0,잠옷용으로 구매했어요,2,16
3,51564285,1,편하고 그냥 입기에도 조아요,2,16
4,46679669,0,잠옷용으로 휘뚜루마뚜루 입으려고 좀 크게 사긴했는데,3,16
5,46679669,1,진짜 많이 커요,3,16
6,46679669,2,조금은 두꺼운 감이 있어요,3,16
7,49013088,0,잠옷으로 입으려고 크게 사긴했는데,2,16
8,49013088,1,정말 크고 두꺼워요,2,16
9,12731504,0,이뻐요.,4,43


## 7-1-6. 분할 결과 검증

In [9]:
# (1) mixed 리뷰(역접 어미 포함) 분할 샘플
sample_mixed = df[
    df['리뷰내용_clean'].str.contains('는데|은데|지만|그런데|근데|하지만|반면', regex=True, na=False)
].sample(5, random_state=42)

print('[mixed 리뷰 분할 샘플]')
for _, row in sample_mixed.iterrows():
    print(f'\n원문 (n={row["n_sents"]}): {row["리뷰내용_clean"][:200]}')
    for i, s in enumerate(row['sentences']):
        print(f'   [{i}] {s}')

[mixed 리뷰 분할 샘플]

원문 (n=5): 처음인데 그래도 잘 맞네요 바지도 이쁘고 허벅지가 커서 허벅지 맞춰서 샀는데 허리는 좀 줄였어요~ 그래도 이쁘네요!
   [0] 처음인데
   [1] 그래도 잘 맞네요
   [2] 바지도 이쁘고 허벅지가 커서 허벅지 맞춰서 샀는데
   [3] 허리는 좀 줄였어요~
   [4] 그래도 이쁘네요!

원문 (n=2): 가성비 바지라서 마감처리나 디테일 적으로 기대 안했는데 깔끔하게 되어있어 놀랐고 너무 마음에 들어요
   [0] 가성비 바지라서 마감처리나 디테일 적으로 기대 안했는데
   [1] 깔끔하게 되어있어 놀랐고 너무 마음에 들어요

원문 (n=4): 생각보다 사이즈, 핏 은 정말 괜찮은데 안에 색감이 생각보다 밝네욤 ㅎㅎ 그래도 정말 맘애 들어서 절입겠습니다아
   [0] 생각보다 사이즈, 핏 은 정말 괜찮은데
   [1] 안에 색감이 생각보다 밝네욤 ㅎㅎ
   [2] 그래도 정말 맘애 들어서 절입겠습니다
   [3] 아

원문 (n=3): 화면보다는 좀더연하고 약간 청녹색? 비슷한느낌나는데 이뻐요~
   [0] 화면보다는 좀더연하고 약간 청녹색?
   [1] 비슷한느낌나는데
   [2] 이뻐요~

원문 (n=4): 너무 스트릿해요 너무 좋아요 색 빠진다고 하는데 없길바래요ㅔ
   [0] 너무 스트릿해요
   [1] 너무 좋아요
   [2] 색 빠진다고 하는데
   [3] 없길바래요ㅔ


In [10]:
# (2) 문장 길이 분포
df_sents['sent_len'] = df_sents['sentence'].str.len()

print('[문장 길이 분포]')
print(df_sents['sent_len'].describe())

n_short = (df_sents['sent_len'] < 5).sum()
n_long  = (df_sents['sent_len'] >= 200).sum()
print(f'\n짧은 문장 (5자 미만): {n_short:,}건 ({n_short/len(df_sents)*100:.2f}%)')
print(f'긴 문장 (200자 이상): {n_long:,}건 ({n_long/len(df_sents)*100:.2f}%)')

if n_short > 0:
    print('\n[짧은 문장 샘플 (감성분석 신뢰도 낮음)]')
    print(df_sents[df_sents['sent_len'] < 5][['sentence']].head(15).to_string(index=False))

[문장 길이 분포]
count    1.531645e+06
mean     1.677030e+01
std      9.997529e+00
min      2.000000e+00
25%      1.000000e+01
50%      1.500000e+01
75%      2.200000e+01
max      4.090000e+02
Name: sent_len, dtype: float64

짧은 문장 (5자 미만): 77,499건 (5.06%)
긴 문장 (200자 이상): 5건 (0.00%)

[짧은 문장 샘플 (감성분석 신뢰도 낮음)]
sentence
    이뻐요.
     좋아요
     좋네요
    크긴해요
     M사도
    이뻐요!
     좋아요
    싶었지만
     했는데
     좋아요
     이뻐요
     이뻐요
    잘샀어요
     크지만
    괜찮네요


In [11]:
# (3) 토픽별 문장 수 분포
topic_sent_counts = df_sents[df_sents['topic_id'] != -1].groupby('topic_id').size().describe()
print('[토픽별 문장 수 분포 (outlier 제외)]')
print(topic_sent_counts)

[토픽별 문장 수 분포 (outlier 제외)]
count       68.000000
mean     15124.970588
std      16407.938181
min       3221.000000
25%       5897.000000
50%       8339.500000
75%      16295.750000
max      83767.000000
dtype: float64


## 7-1-7. 저장

In [12]:
output_path = OUTPUT_DIR + 'step7_1_sentences.parquet'
df_sents.to_parquet(output_path, index=False)

print(f'저장 완료: {output_path}')
print(f'  총 {len(df_sents):,}문장')
print(f'  메모리: {df_sents.memory_usage(deep=True).sum() / 1024**2:.1f} MB')

print(f'\n[7-2(측면 매칭)에서 사용할 핵심 컬럼]')
print(f'  - 리뷰번호, sent_id, sentence  (분석 대상)')
print(f'  - topic_id, 브랜드, 카테고리   (집계용)')
print(f'  - sent_len                    (신뢰도 가중치용)')

저장 완료: /content/drive/MyDrive/sparta/tp4/review_analysis/data/step7_1_sentences.parquet
  총 1,531,645문장
  메모리: 267.8 MB

[7-2(측면 매칭)에서 사용할 핵심 컬럼]
  - 리뷰번호, sent_id, sentence  (분석 대상)
  - topic_id, 브랜드, 카테고리   (집계용)
  - sent_len                    (신뢰도 가중치용)
